# AgentRegistry, end to end: one agent on kagent **and** AWS Bedrock AgentCore

**Persona:** a **developer** taking an agent from nothing to running in production. Run the cells top to bottom; each one runs a real command and shows its output. The markdown before each step says *what* it does and *why* it matters.

**The problem we're showing.** An agent on a laptop is a science project. An agent a whole organisation can *find, govern, and run anywhere* is a platform. That needs three things a laptop never gives you: a **catalog** of what exists, **packaging** that makes each piece reproducible, and a **runtime** that hosts the result behind real auth. That is exactly what AgentRegistry provides, and **`arctl`** is its CLI.

**The punchline.** We scaffold an agent (plus an MCP tool server and a reusable skill), publish them to the catalog as OCI images, then deploy **the same catalog agent to two completely different runtimes** — Solo Enterprise for **kagent** in a local Kubernetes cluster, and **AWS Bedrock AgentCore** — by changing **one line**: the Deployment's `runtimeRef`. Same image, same model (Anthropic Claude), two clouds.

```mermaid
flowchart LR
  Eng[Field engineer<br/>arctl] -->|init / build / apply| Cat[(AgentRegistry catalog<br/>summarizer · textkit · summary-style)]
  Cat -->|Deployment<br/>runtimeRef: kind-kagent| K[kagent on kind<br/>OIDC enforced]
  Cat -->|Deployment<br/>runtimeRef: aws-agentcore| A[AWS Bedrock<br/>AgentCore]
  classDef c fill:#7C3AED,color:#fff,stroke:#5B21B6,stroke-width:2px
  classDef r fill:#D1FAE5,stroke:#10B981,color:#064E3B,stroke-width:2px
  class Cat c
  class K,A r
```

This is a **bash-kernel** notebook ([`bash_kernel`](https://github.com/takluyver/bash_kernel)). The local half (kagent) needs only Docker + an Anthropic key + a Solo license. The AWS half (AgentCore) is an optional add-on that needs an AWS account.


## 0. Credentials — secure, but simple

No keys go in notebook cells (they'd end up in git). Instead they live in a gitignored **`.env.local`** you fill in once. The cell below creates it from `env.example` on first run — **edit `.env.local`, then re-run this cell.** It prints only whether each value is *set* and its length, never the value.

- `ANTHROPIC_API_KEY`, `SOLO_LICENSE_KEY` — required for the kagent half.
- `AWS_PROFILE` / `AWS_REGION` — only for the AgentCore half.
- Keep Solo creds in an existing env file already? Put `SECRETS_FILE=/path/to/it` in `.env.local` and it's sourced too.


In [ ]:
if [ ! -f .env.local ]; then
  cp env.example .env.local
  echo "Created .env.local — edit it with your keys, then RE-RUN this cell."
fi

set -a
[ -f .env.local ] && . ./.env.local
[ -n "${SECRETS_FILE:-}" ] && [ -f "$SECRETS_FILE" ] && . "$SECRETS_FILE"
set +a

export PATH="$HOME/.arctl/bin:$PATH"
export CLUSTER_NAME="${CLUSTER_NAME:-agentcore-demo}"
export ARCTL_API_BASE_URL="${ARCTL_API_BASE_URL:-http://localhost:12121}"
export DOCKER_REPO="${DOCKER_REPO:-solo-public/agentregistry-enterprise}"
export OIDC_AUTO_AUTH_ENABLED="${OIDC_AUTO_AUTH_ENABLED:-true}"

mask() { if [ -n "$1" ]; then echo "set (${#1} chars)"; else echo "MISSING"; fi; }
echo "ANTHROPIC_API_KEY : $(mask "${ANTHROPIC_API_KEY:-}")"
echo "SOLO_LICENSE_KEY  : $(mask "${SOLO_LICENSE_KEY:-}")"
echo "AWS_PROFILE       : ${AWS_PROFILE:-<unset> (only needed for AgentCore)}"
echo "AWS_REGION        : ${AWS_REGION:-us-east-1}"
echo "arctl             : $(arctl version 2>/dev/null | awk '/arctl version/{print $3}' || echo '<run step 1>')"
echo "cluster           : kind-$CLUSTER_NAME"


## 1. Prerequisites

**What:** install/validate the CLIs and pin the enterprise **`arctl`** to `v2026.5.4`.

**Why that version:** `arctl` v2026.6.x moved the registry server onto a cluster and dropped the local `daemon`. For a laptop demo we want the standalone daemon, so we pin the latest line that still ships it.


In [ ]:
./scripts/00-prereqs.sh


## 2. Bring up the platform (one-time, ~15 min)

**What:** four idempotent steps that stand up everything the agent will run on. This is plumbing, not the teaching moment, so it's scripted — but here's what each does and why:

1. **kind cluster + a local OCI registry** (`localhost:5001`) — the registry `arctl build --push` pushes to and the cluster pulls from.
2. **Keycloak + the `solo` realm** — Solo Enterprise for kagent enforces OIDC; this is the identity provider it validates tokens against. *(This is why the agent is protected the moment it lands.)*
3. **Solo Enterprise for kagent** — the Kubernetes runtime that will host the agent, with Anthropic as the model provider.
4. **the `arctl daemon`** — the AgentRegistry control plane and catalog, running standalone in Docker with its embedded auth (catalog + web UI on `http://localhost:12121`).

Re-running is safe; finished steps short-circuit.


In [ ]:
./scripts/01-cluster.sh && ./scripts/02-keycloak.sh && ./scripts/03-kagent.sh && ./scripts/04-daemon.sh


**Connect `arctl` to the catalog.** The daemon's embedded IdP issues an admin bearer; we mint one and keep it for the rest of the session. `arctl get runtimes` proves the CLI is talking to the control plane (the daemon seeds a couple of default runtimes).


In [ ]:
export ARCTL_API_TOKEN="$(curl -s -X POST "$ARCTL_API_BASE_URL/api/autoauth/oauth/token" \
  -H 'Content-Type: application/x-www-form-urlencoded' \
  -d 'grant_type=client_credentials&client_id=admin&scope=openid profile email Groups' | jq -r .access_token)"
echo "token: $([ -n "$ARCTL_API_TOKEN" ] && echo minted || echo FAILED)"
arctl user whoami
arctl get runtimes


## 3. Scaffold the building blocks with `arctl init`

**What:** `arctl init` generates a complete, runnable project — source layout, manifest, Dockerfile, env wiring, tests — for each artifact kind. **Why it matters:** this is the day of boilerplate a team *doesn't* write; the framework, language, and model are chosen up front and the pieces are wired to each other from the first run.

The demo ships three artifacts, already scaffolded **and** customised, under `artifacts/`:

| Kind | Name | What it is |
|------|------|-----------|
| MCPServer | `acme/textkit` | tools `word_count`, `extract_links` (FastMCP, Python) |
| Skill | `summary-style` | a `SKILL.md` house format, baked into the agent at build |
| Agent | `summarizer` | ADK Python, **Anthropic `claude-haiku-4-5`**, uses textkit + the skill |

Run the cell to see `arctl init` build a fresh agent from nothing (into a scratch dir), then we use the committed, customised ones.


In [ ]:
# The commands that produced artifacts/ (then customised with real tools + SKILL.md):
cat <<'CMDS'
  arctl init mcp   acme/textkit  --framework fastmcp --language python
  arctl init skill summary-style
  arctl init agent summarizer    --framework adk --language python \
      --model-provider anthropic --model-name claude-haiku-4-5 --local-mcp ./textkit
CMDS
echo
# Watch arctl scaffold a brand-new agent live (throwaway dir — does not touch artifacts/):
rm -rf /tmp/arctl-scratch && mkdir -p /tmp/arctl-scratch
( cd /tmp/arctl-scratch && arctl init agent demoagent \
    --framework adk --language python \
    --model-provider anthropic --model-name claude-haiku-4-5 >/dev/null )
echo "arctl generated:"
find /tmp/arctl-scratch/demoagent -maxdepth 2 -type f | sed 's#/tmp/arctl-scratch/##' | sort


**Make the scaffold real.** The three customisations that turn templates into the demo: a tool is just a decorated function the loader discovers by filename; the skill is plain instructions-as-an-artifact; the agent folds the skill into its instruction and reads the MCP tools from env. Peek at each:


In [ ]:
echo '===== a textkit tool (one file = one tool) =====';        sed -n '1,20p' artifacts/textkit/src/tools/word_count.py
echo; echo '===== the skill (house format) =====';               sed -n '1,18p' artifacts/summary-style/SKILL.md
echo; echo '===== the agent wiring (model + skill + MCP tools) ====='; grep -nE 'LiteLlm|root_agent|instruction|get_mcp_tools|load_baked' artifacts/summarizer/summarizer/agent.py | head


## 4. Prove it locally with `arctl run` (no cluster)

**What/why:** before any cluster, `arctl run` is the inner dev loop — it builds the agent image, starts it, and drops you into an interactive A2A chat, with the MCP server running alongside on Docker. The full agent→tools path, on a laptop, in one command.

This one is interactive, so run it in a **terminal** (not inline):

```sh
./scripts/test-local.sh
# then type:  summarize this: <paste a paragraph with a couple of https:// links>
# Ctrl-C to exit
```


## 5. Build the images and publish to the catalog

**What:** `arctl build --push` builds each artifact's Dockerfile and pushes to the local registry; `arctl apply` registers it in the catalog. **Why it matters:** from here the catalog is the shared source of truth — anyone pointed at the registry can discover what exists and where it came from, and each entry resolves to a real OCI image. The skill is git-sourced, so it has no image.


In [ ]:
arctl build ./artifacts/textkit    --push   # -> localhost:5001/textkit:latest
arctl build ./artifacts/summarizer --push   # -> localhost:5001/summarizer:latest


In [ ]:
# Publish all three. Order matters once: the MCPServer must exist before the
# Agent that references it (the registry resolves spec.mcpServers at apply time).
arctl apply -f artifacts/textkit/mcp.yaml
arctl apply -f artifacts/summary-style/skill.yaml
arctl apply -f artifacts/summarizer/agent.yaml
echo; echo '===== the catalog ====='
arctl get mcp acme/textkit; arctl get skill summary-style; arctl get agent summarizer


## 6. Point the registry at the cluster — a Kubernetes **Runtime**

**What:** a `Runtime` is the registry's pointer to somewhere agents can run. For kagent it's a namespace + a kubeconfig.

**Why the extra plumbing:** the daemon runs *inside* Docker, so `127.0.0.1` in a normal kubeconfig would mean the daemon's own container, not your cluster. We join the daemon to the kind network and hand it the **internal** kubeconfig, whose API-server address is the control-plane container's hostname (and whose TLS matches it). This is the one laptop-specific wrinkle; on a real cluster the kubeconfig just works.


In [ ]:
DAEMON_CTR=$(docker ps --filter publish=12121 --format '{{.Names}}' | head -1)
docker network connect kind "$DAEMON_CTR" 2>/dev/null || true
echo "daemon '$DAEMON_CTR' on the kind network"

cat > /tmp/runtime-kagent.yaml <<EOF
apiVersion: ar.dev/v1alpha1
kind: Runtime
metadata:
  name: kind-kagent
spec:
  type: Kubernetes
  config:
    namespace: kagent
    kubeconfig: |
$(kind get kubeconfig --internal --name "$CLUSTER_NAME" | sed 's/^/      /')
EOF
arctl apply -f /tmp/runtime-kagent.yaml
arctl get runtimes


## 7. Deploy the agent onto kagent (runtime #1)

**What:** a `Deployment` binds an Agent to a Runtime and carries per-instance env. Note `runtimeRef.name: kind-kagent` — **remember this line; it's the only thing that changes for AWS later.** The registry's Kubernetes adapter translates this into kagent CRDs (a BYO Agent + a kmcp MCPServer) and the controller schedules them.


In [ ]:
echo '===== the Deployment we are applying ====='; cat yaml/deployment.yaml
echo; echo '===== applying ====='
envsubst < yaml/deployment.yaml | arctl apply -f -

# The OCI/stdio MCP runs via a kmcp relay that launches it by an explicit
# command; set that so the relay can start the FastMCP server in-container.
echo 'waiting for the textkit kmcp MCPServer to appear...'
for _ in $(seq 1 40); do
  MCPSRV=$(kubectl --context kind-$CLUSTER_NAME -n kagent get mcpserver -o name 2>/dev/null | grep -i textkit | head -1)
  [ -n "$MCPSRV" ] && break; sleep 3
done
[ -n "$MCPSRV" ] && kubectl --context kind-$CLUSTER_NAME -n kagent patch "$MCPSRV" --type merge \
  -p '{"spec":{"deployment":{"cmd":"python","args":["src/main.py"]}}}' && echo "set $MCPSRV cmd"
arctl get deployments


In [ ]:
# Give the controller a moment, then watch the agent + MCP land as kagent CRDs:
kubectl --context kind-$CLUSTER_NAME -n kagent get agents,pods | grep -vi kmcp-enterprise


## 8. Talk to the hosted agent — through real OIDC

**Why this is an honest test:** the agent sits behind Solo Enterprise for kagent's OIDC interceptor, so we don't get to skip auth. `ask.sh` mints a real Keycloak token for **alice** (group `field-fte`, mapped to a kagent Admin role so she may invoke agents), then POSTs an A2A `message/send` to the controller. It prints alice's token claims and the reply.

Watch the reply: the **bold headline + bullets** come from the *skill*, the proportional length from the `word_count` *MCP tool*, and the `Sources:` line from `extract_links` — every artifact in the catalog, plus the runtime's auth, in one request.

(`ask.sh` handles the port-forwards and token mint; it runs this against the controller's `/api/a2a/<ns>/<agent>/` endpoint.)


In [ ]:
./scripts/ask.sh "summarize this: AgentRegistry is an open catalog for agents, MCP servers and skills. The arctl CLI scaffolds an artifact, builds it into an OCI image, and publishes it so others can reuse it. A Kubernetes Runtime adapter turns a Deployment into kagent CRDs. Docs at https://aregistry.ai and source at https://github.com/agentregistry-dev/agentregistry."


Optional — open the kagent dashboard and chat there too:

```sh
./scripts/port-forward.sh   # http://localhost:8080
```

---
# The punchline: the **same** agent on AWS Bedrock AgentCore (runtime #2)

Everything below deploys the **identical published `summarizer`** to AWS — no rebuild of the agent's logic, no second codebase. We register a second Runtime (type `BedrockAgentCore`) and a second Deployment whose only meaningful difference is `runtimeRef`. **Needs an AWS account; skip for a local-only demo.**


## 9. Sign in to AWS

Set `AWS_PROFILE` in `.env.local` and re-run the Setup cell, then sign in. `aws sso login` opens your browser; nothing here prints your account or role. The cell also hands those credentials to the arctl daemon (it restarts it) so the daemon can assume the cross-account role when it deploys to AgentCore. Run it in a terminal if the browser doesn't pop from the notebook:

```sh
aws sso login --profile "$AWS_PROFILE"
```


In [ ]:
if [ -z "${AWS_PROFILE:-}" ]; then echo "Set AWS_PROFILE in .env.local and re-run the Setup cell first."; else
  aws sts get-caller-identity >/dev/null 2>&1 || aws sso login --profile "$AWS_PROFILE"
  export AWS_REGION="${AWS_REGION:-us-east-1}"
  export AWS_ACCOUNT_ID="$(aws sts get-caller-identity --query Account --output text)"
  echo "AWS session live — account ****${AWS_ACCOUNT_ID: -4} / region $AWS_REGION"

  # The arctl daemon (a Docker container) is what assumes the cross-account role
  # to manage AgentCore, so it needs AWS creds. Its compose forwards AWS_* from
  # the environment, but it was started in step 2 — before you logged in. Resolve
  # creds and restart it so it picks them up (the catalog persists in postgres
  # across the restart).
  eval "$(aws configure export-credentials --format env)"
  export AWS_ACCESS_KEY_ID AWS_SECRET_ACCESS_KEY AWS_SESSION_TOKEN
  arctl daemon stop >/dev/null 2>&1; arctl daemon start >/dev/null 2>&1
  DC=$(docker ps --filter publish=12121 --format '{{.Names}}' | head -1)
  docker network connect kind "$DC" 2>/dev/null || true
  sleep 5
  export ARCTL_API_TOKEN="$(curl -s -X POST "$ARCTL_API_BASE_URL/api/autoauth/oauth/token" \
    -H 'Content-Type: application/x-www-form-urlencoded' \
    -d 'grant_type=client_credentials&client_id=admin&scope=openid profile email Groups' | jq -r .access_token)"
  echo "daemon restarted with AWS credentials"
fi

## 10. Grant AgentRegistry access to your AWS account

**What:** `arctl runtime setup bedrock-agent-core` generates a CloudFormation template that creates a cross-account IAM role letting AgentRegistry manage AgentCore in *your* account, plus an **External ID** (a shared secret that scopes the trust). This step makes **no AWS changes** — it just prints the template and the ID. We then deploy the stack and read back the role ARN.


In [ ]:
mkdir -p .agentcore
arctl runtime setup bedrock-agent-core --aws-account-id "$AWS_ACCOUNT_ID" \
  --role-name AgentRegistryAccessRole-agentcore-demo \
  2> >(tee .agentcore/setup.stderr >&2) > .agentcore/cf.yaml
# The External ID is a base64url token (not a UUID):
export AWS_EXTERNAL_ID=$(grep -ioE 'External ID:[[:space:]]*[A-Za-z0-9_-]+' .agentcore/setup.stderr | awk '{print $NF}' | head -1)
echo "External ID parsed: $([ -n "$AWS_EXTERNAL_ID" ] && echo yes || echo NO)"
echo "CF template: $(wc -l < .agentcore/cf.yaml) lines -> .agentcore/cf.yaml"


In [ ]:
# Deploy the stack (idempotent) and read the role ARN it outputs.
if aws cloudformation describe-stacks --stack-name AgentRegistryAccess >/dev/null 2>&1; then
  echo 'stack already exists — reusing'
else
  aws cloudformation create-stack --stack-name AgentRegistryAccess \
    --template-body file://.agentcore/cf.yaml --capabilities CAPABILITY_NAMED_IAM
  echo 'waiting for stack create...'; aws cloudformation wait stack-create-complete --stack-name AgentRegistryAccess
fi
export AWS_ROLE_ARN=$(aws cloudformation describe-stacks --stack-name AgentRegistryAccess \
  --query 'Stacks[0].Outputs[?OutputKey==`RoleArn`].OutputValue' --output text)
echo "role ARN: ${AWS_ROLE_ARN%%:role*}:role/****"


## 11. Register the AgentCore **Runtime**

**What:** the second `Runtime` — same kind of object as the kagent one, just `type: BedrockAgentCore` pointing at your AWS account via the role + External ID. After this, `arctl get runtimes` shows both: one Kubernetes, one AWS.


In [ ]:
cat > /tmp/runtime-aws.yaml <<EOF
apiVersion: ar.dev/v1alpha1
kind: Runtime
metadata:
  name: aws-agentcore
spec:
  type: BedrockAgentCore
  config:
    roleArn: "${AWS_ROLE_ARN}"
    externalId: "${AWS_EXTERNAL_ID}"
    region: "${AWS_REGION}"
EOF
arctl apply -f /tmp/runtime-aws.yaml
arctl get runtimes


## 12. Push the image to ECR and point the Agent at it

**Why:** AgentCore can't pull from your laptop's `localhost:5001`, and it clones the agent **source** from git at deploy time. So we push the agent image to **ECR** and update the catalog Agent to reference the ECR image + the git repo. (`--platform linux/amd64` because AgentCore wants amd64; on Apple Silicon this cross-builds.)


In [ ]:
export ECR_HOST="${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_REGION}.amazonaws.com"
export ECR_IMAGE="${ECR_HOST}/summarizer:0.0.1"
aws ecr describe-repositories --repository-names summarizer >/dev/null 2>&1 \
  || aws ecr create-repository --repository-name summarizer >/dev/null
aws ecr get-login-password --region "$AWS_REGION" | docker login --username AWS --password-stdin "$ECR_HOST" >/dev/null
arctl build ./artifacts/summarizer --push --platform linux/amd64 --image "$ECR_IMAGE"
echo "pushed $ECR_IMAGE"


In [ ]:
# Re-publish the Agent with the cloud-reachable image + git source. Set
# AGENT_GIT_* in .env.local to a repo/branch/subfolder AWS can clone.
cat > /tmp/agent-aws.yaml <<EOF
apiVersion: ar.dev/v1alpha1
kind: Agent
metadata:
  name: summarizer
spec:
  description: Summarizes pasted text in the house format, using textkit MCP tools.
  modelName: claude-haiku-4-5
  modelProvider: anthropic
  source:
    image: ${ECR_IMAGE}
    repository:
      url: ${AGENT_GIT_URL:?set AGENT_GIT_URL in .env.local (a repo AWS can clone)}
      branch: ${AGENT_GIT_BRANCH:-main}
      subfolder: ${AGENT_GIT_SUBFOLDER:-agentregistry-agentcore-kind/artifacts/summarizer}
EOF
arctl apply -f /tmp/agent-aws.yaml


## 13. Deploy the same agent to AgentCore (runtime #2)

**The whole point in one cell.** Same Agent, same model — only `runtimeRef` differs (`aws-agentcore` instead of `kind-kagent`). The Anthropic key rides along as env so the container can call the model from inside AWS.


In [ ]:
cat > /tmp/deployment-aws.yaml <<EOF
apiVersion: ar.dev/v1alpha1
kind: Deployment
metadata:
  name: summarizer-agentcore
spec:
  targetRef:
    kind: Agent
    name: summarizer
  runtimeRef:
    kind: Runtime
    name: aws-agentcore          # <-- the only meaningful change vs runtime #1
  runtimeConfig:
    region: ${AWS_REGION}
  env:
    ANTHROPIC_API_KEY: "${ANTHROPIC_API_KEY}"
EOF
arctl apply -f /tmp/deployment-aws.yaml
arctl get deployments


## 14. Test the agent on AgentCore

Open **Amazon Bedrock → AgentCore** in the AWS console, find your runtime, and send a JSON-RPC `message/send` payload in the playground:

```json
{
  "jsonrpc": "2.0", "id": "req-001", "method": "message/send",
  "params": { "message": {
    "role": "user", "messageId": "12345678-1234-1234-1234-123456789012",
    "parts": [{"kind": "text", "text": "summarize this: <paste a paragraph with a couple of https:// links>"}]
  }}
}
```

**The takeaway:** one catalog entry, governed in one place, ran unchanged on Kubernetes *and* on AWS-managed infrastructure. As a developer, moving or multi-homing an agent is a one-line `runtimeRef` change, not a porting project.


## 15. Teardown

Removes the AWS AgentCore bits (CloudFormation stack, runtime, ECR repo, deployment) and the local platform (kind cluster, daemon, registry). Both forms are safe to re-run; the AWS path no-ops cleanly without a live AWS session.

```sh
./scripts/cleanup.sh agentcore   # AWS only
./scripts/cleanup.sh             # everything
```


In [ ]:
# Uncomment to tear everything down:
# ./scripts/cleanup.sh
